# 06 — Outlier Detection
## Guided Jupyter Notebook

This notebook demonstrates how to flag unusual numerical observations using the IQR rule and z-scores, visualize them, and compare possible treatments.

**Important:** Detection does not prove a value is wrong. Inspect flagged records before making changes.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
pd.set_option("display.max_columns", None)


## 2. Generate a reproducible transaction dataset
Several unusual values are deliberately included. They are practice examples, not automatically invalid transactions.

In [ ]:
rng = np.random.default_rng(61)
n = 180

amounts = rng.lognormal(mean=3.7, sigma=0.55, size=n)
items = rng.poisson(lam=3.5, size=n) + 1
ages = np.clip(rng.normal(38, 12, size=n), 18, 80).round().astype(int)

df = pd.DataFrame({
    "transaction_id": [f"TXN-{i:04d}" for i in range(1, n + 1)],
    "transaction_amount": amounts.round(2),
    "items_count": items,
    "customer_age": ages,
    "region": rng.choice(["North", "South", "East", "West"], size=n),
    "payment_method": rng.choice(["Card", "UPI", "Cash", "Wallet"], size=n)
})

# Inject intentionally unusual values for practice.
df.loc[4, "transaction_amount"] = 2500.00
df.loc[27, "transaction_amount"] = 0.50
df.loc[27, "items_count"] = 1
df.loc[72, "items_count"] = 35
df.loc[111, "customer_age"] = 99
df.loc[rng.choice(df.index, 4, replace=False), "transaction_amount"] = np.nan

df.head()


## 3. Inspect data quality and distributions

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.rename("dtype").to_frame())
display(df.isna().sum().to_frame("missing_count"))
display(df.describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="transaction_amount", bins=20, kde=True, ax=axes[0])
axes[0].set_title("Transaction Amount Distribution")
sns.boxplot(data=df, x="transaction_amount", ax=axes[1])
axes[1].set_title("Transaction Amount Box Plot")
plt.tight_layout()
plt.show()


## 4. Detect potential outliers with IQR

In [ ]:
def iqr_bounds(series, factor=1.5):
    valid = series.dropna()
    if valid.empty:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    q1, q3 = valid.quantile(0.25), valid.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - factor * iqr, q3 + factor * iqr

q1, q3, iqr, lower, upper = iqr_bounds(df["transaction_amount"])
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"Lower fence={lower:.2f}, Upper fence={upper:.2f}")

iqr_mask = df["transaction_amount"].notna() & (
    (df["transaction_amount"] < lower) |
    (df["transaction_amount"] > upper)
)
print("IQR-flagged count:", int(iqr_mask.sum()))
print("IQR-flagged percentage:", round(iqr_mask.sum() / df["transaction_amount"].notna().sum() * 100, 2), "%")
display(df.loc[iqr_mask])


## 5. Detect potential outliers with z-scores
A common screening convention is absolute z-score greater than 3. This can be unreliable for skewed distributions, so compare it with IQR flags and plots.

In [ ]:
s = df["transaction_amount"]
std = s.std(ddof=1)
if pd.notna(std) and std > 0:
    z_scores = (s - s.mean()) / std
    z_mask = s.notna() & (z_scores.abs() > 3)
else:
    z_scores = pd.Series(np.nan, index=s.index)
    z_mask = pd.Series(False, index=s.index)

print("Z-score flagged count:", int(z_mask.sum()))
display(df.loc[z_mask])
display(pd.DataFrame({"iqr_flag": iqr_mask, "zscore_flag": z_mask}).value_counts().to_frame("row_count"))


## 6. Compare methods across numeric columns

In [ ]:
def iqr_summary(series, factor=1.5):
    valid = series.dropna()
    if valid.empty:
        return {"count": 0, "outlier_count": 0, "outlier_percent": np.nan}
    q1, q3 = valid.quantile(0.25), valid.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    mask = (valid < lower) | (valid > upper)
    return {"count": int(valid.count()), "q1": q1, "q3": q3, "iqr": iqr,
            "lower_fence": lower, "upper_fence": upper,
            "outlier_count": int(mask.sum()), "outlier_percent": float(mask.mean() * 100)}

numeric_cols = ["transaction_amount", "items_count", "customer_age"]
summary_table = pd.DataFrame({col: iqr_summary(df[col]) for col in numeric_cols}).T
display(summary_table.round(2))


## 7. Investigate flagged records

In [ ]:
display(df.loc[iqr_mask, [
    "transaction_id", "transaction_amount", "items_count",
    "customer_age", "region", "payment_method"
]])

sns.scatterplot(data=df, x="items_count", y="transaction_amount", hue="region")
plt.title("Transaction Amount vs Item Count")
plt.xlabel("Number of items")
plt.ylabel("Transaction amount")
plt.tight_layout()
plt.show()


## 8. Compare possible treatments
Create separate copies. Capping changes values; filtering removes rows. Neither action should be automatic.

In [ ]:
capped_df = df.copy()
capped_df["transaction_amount"] = capped_df["transaction_amount"].clip(lower=lower, upper=upper)
filtered_df = df.loc[~iqr_mask].copy()

print("Original row count:", len(df))
print("Capped row count:", len(capped_df))
print("Filtered row count:", len(filtered_df))

comparison = pd.DataFrame({
    "original": df["transaction_amount"].describe(),
    "capped": capped_df["transaction_amount"].describe(),
    "filtered": filtered_df["transaction_amount"].describe()
})
display(comparison.round(2))


## 9. Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=df, x="transaction_amount", bins=20, ax=axes[0])
axes[0].set_title("Original")
sns.histplot(data=capped_df, x="transaction_amount", bins=20, ax=axes[1])
axes[1].set_title("Capped at IQR Fences")
sns.histplot(data=filtered_df, x="transaction_amount", bins=20, ax=axes[2])
axes[2].set_title("Rows Filtered by IQR Rule")
plt.tight_layout()
plt.show()


## 10. Reflection

1. Which values were flagged by IQR and z-score methods?
2. Why can the methods produce different flags?
3. Which flagged records seem plausible, and what would you verify?
4. What changed after capping or filtering?
5. Why should raw data remain unchanged?
6. How should thresholds be handled in a machine-learning workflow?

Continue with `practice.md`, then check your reasoning in `solution.md`.


In [ ]:
print("IQR flags:", int(iqr_mask.sum()))
print("Z-score flags:", int(z_mask.sum()))
print("Overlap:", int((iqr_mask & z_mask).sum()))
print("Original amount median:", round(df["transaction_amount"].median(), 2))
print("Capped amount median:", round(capped_df["transaction_amount"].median(), 2))
print("Filtered amount median:", round(filtered_df["transaction_amount"].median(), 2))
